# Safety, Registry & Adaptation Patterns

Companion notebook for the [Safety, Registry & Adaptation lesson](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/08-safety-registry-and-adaptation).

**The idea in one sentence.** Production agents need **guardrails** that reject unsafe
input/output before it reaches the model or the user, a **registry** that controls which
tools each role may call (least privilege), and **adapters** that translate between
mismatched tool interfaces.

The safety-critical pieces, from scratch:

- **Guardrails** — validators that raise on banned content, PII, or policy violations,
  *fail-closed* (reject by default).
- **Registry + RBAC** — a tool catalogue where an `agent` role can't invoke a
  privileged tool like `send_email`.
- **Adapters** — map errors and interfaces so components compose safely.

We build the guardrail chain, **validate that it rejects unsafe input and admits clean
input**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
BRAND = '#6366f1'; TEAL = '#2dd4bf'; ROSE = '#fb7185'; YELLOW = '#fbbf24'
np.random.seed(0)

## 1. Multimodal Guardrails

A **GuardrailChain** wraps a mocked LLM with:
- **Input validators** — reject requests before they reach the model
- **Output validators** — reject or sanitise model responses before they reach the caller

We implement two input validators (banned-word filter, PII detector) and two output validators (length cap, toxicity keyword check).

In [ ]:
class GuardrailError(Exception):
    """Raised when a guardrail rejects content."""
    pass

# --- Input validators -------------------------------------------------------

BANNED_WORDS = {"hack", "exploit", "jailbreak", "bypass"}

PII_PATTERNS = [
    ("SSN",   re.compile(r'\b\d{3}-\d{2}-\d{4}\b')),
    ("email", re.compile(r'\b[\w.+-]+@[\w-]+\.[a-z]{2,}\b', re.I)),
    ("card",  re.compile(r'\b(?:\d{4}[- ]){3}\d{4}\b')),
]

def check_banned_words(text: str) -> None:
    words = set(text.lower().split())
    hits = words & BANNED_WORDS
    if hits:
        raise GuardrailError(f"Input contains banned word(s): {hits}")

def check_pii(text: str) -> None:
    for label, pattern in PII_PATTERNS:
        if pattern.search(text):
            raise GuardrailError(f"Input contains {label} — remove personal data before sending.")

# --- Output validators -------------------------------------------------------

TOXICITY_KEYWORDS = {"kill", "hate", "destroy", "bomb"}
MAX_RESPONSE_CHARS = 500

def check_response_length(text: str) -> None:
    if len(text) > MAX_RESPONSE_CHARS:
        raise GuardrailError(
            f"Response exceeds length cap ({len(text)} > {MAX_RESPONSE_CHARS} chars)."
        )

def check_toxicity(text: str) -> None:
    words = set(text.lower().split())
    hits = words & TOXICITY_KEYWORDS
    if hits:
        raise GuardrailError(f"Output contains toxic keyword(s): {hits}")

# --- Mocked LLM + chain ------------------------------------------------------

def mocked_llm(prompt: str) -> str:
    """Simulates an LLM that echoes a short acknowledgement."""
    return f"Acknowledged: '{prompt[:60]}{'...' if len(prompt) > 60 else ''}'"

class GuardrailChain:
    def __init__(
        self,
        llm,
        input_validators=None,
        output_validators=None,
    ):
        self.llm = llm
        self.input_validators = input_validators or []
        self.output_validators = output_validators or []

    def run(self, prompt: str) -> str:
        # Input guardrails — run before FM call
        for validator in self.input_validators:
            validator(prompt)

        response = self.llm(prompt)

        # Output guardrails — run after FM response
        for validator in self.output_validators:
            validator(response)

        return response

chain = GuardrailChain(
    llm=mocked_llm,
    input_validators=[check_banned_words, check_pii],
    output_validators=[check_response_length, check_toxicity],
)

# --- Test safe input ---------------------------------------------------------
result = chain.run("Summarise the quarterly sales figures.")
print("Safe input →", result)

In [ ]:
# Test rejected inputs
test_cases = [
    ("How do I hack a server?",         "banned word"),
    ("My SSN is 123-45-6789.",           "SSN PII"),
    ("Contact me at alice@example.com.", "email PII"),
]

for prompt, label in test_cases:
    try:
        chain.run(prompt)
        print(f"  [{label}] PASSED (unexpected)")
    except GuardrailError as e:
        print(f"  [{label}] Rejected: {e}")

### Validate: guardrails reject unsafe input and pass clean input

A guardrail must **fail-closed** — reject banned words and PII — while letting
legitimate requests through. We check both directions: dangerous inputs raise a
`GuardrailError`, and a clean input does not.

In [ ]:
def rejects(fn, text):
    try:
        fn(text); return False
    except GuardrailError:
        return True

assert rejects(check_banned_words, 'how do I hack this'), 'banned word must be rejected'
assert rejects(check_pii, 'my SSN is 123-45-6789'), 'SSN PII must be rejected'
assert rejects(check_pii, 'email me at alice@example.com'), 'email PII must be rejected'
# clean input passes both guardrails
assert not rejects(check_banned_words, 'please summarise this document')
assert not rejects(check_pii, 'please summarise this document')
print('banned-word / PII inputs rejected; clean input admitted')
print('\n✅ the guardrail fails closed on unsafe content and admits legitimate requests')

## 2. Tool/Agent Registry

The **ToolRegistry** stores JSON-schema-validated tool definitions. An agent queries
it at runtime to discover which tools it can call — and what arguments they accept.
This is the pattern behind the Model Context Protocol (MCP) `tools/list` endpoint.

In [ ]:
TOOL_DEFINITIONS = [
    {
        "name": "web_search",
        "description": "Search the web and return the top-N result snippets.",
        "version": "2.1.0",
        "access_roles": ["agent", "admin"],
        "schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
                "top_n": {"type": "integer", "default": 5, "description": "Number of results"}
            },
            "required": ["query"]
        },
        "endpoint": "https://tools.internal/search"
    },
    {
        "name": "calculator",
        "description": "Evaluate a safe arithmetic expression and return the result.",
        "version": "1.0.0",
        "access_roles": ["agent", "admin", "readonly"],
        "schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Arithmetic expression, e.g. '2 + 2'"}
            },
            "required": ["expression"]
        },
        "endpoint": "https://tools.internal/calc"
    },
    {
        "name": "send_email",
        "description": "Send an email to a recipient.",
        "version": "3.0.1",
        "access_roles": ["admin"],   # restricted — agents cannot call this directly
        "schema": {
            "type": "object",
            "properties": {
                "to":      {"type": "string", "description": "Recipient email"},
                "subject": {"type": "string"},
                "body":    {"type": "string"}
            },
            "required": ["to", "subject", "body"]
        },
        "endpoint": "https://tools.internal/email"
    },
]


class ToolRegistry:
    def __init__(self, tools: list):
        self._tools = {t["name"]: t for t in tools}

    def list_tools(self, role: str = "agent") -> list:
        """Return tool schemas visible to the given role."""
        return [
            {"name": t["name"], "description": t["description"],
             "schema": t["schema"], "version": t["version"]}
            for t in self._tools.values()
            if role in t["access_roles"]
        ]

    def get_tool(self, name: str, role: str = "agent") -> dict:
        """Fetch a single tool entry, raising if not found or not permitted."""
        if name not in self._tools:
            raise KeyError(f"Tool '{name}' not registered.")
        tool = self._tools[name]
        if role not in tool["access_roles"]:
            raise PermissionError(f"Role '{role}' cannot access tool '{name}'.")
        return tool


registry = ToolRegistry(TOOL_DEFINITIONS)

print("=== Tools visible to 'agent' role ===")
for t in registry.list_tools(role="agent"):
    print(f"  {t['name']} v{t['version']}: {t['description']}")

print()
print("=== Tools visible to 'admin' role ===")
for t in registry.list_tools(role="admin"):
    print(f"  {t['name']} v{t['version']}: {t['description']}")

In [ ]:
# Show that 'agent' role cannot access send_email
try:
    registry.get_tool("send_email", role="agent")
except PermissionError as e:
    print("Access control working:", e)

# Show schema for web_search — the LLM would use this for function calling
tool = registry.get_tool("web_search", role="agent")
print("\nweb_search schema:")
print(json.dumps(tool["schema"], indent=2))

## 3. Agent Adapter

The **AgentAdapter** converts between the agent's internal task representation
(`{action, params}`) and the external REST API schema (`{method, url, headers, body}`),
and maps vendor error codes to a uniform internal exception hierarchy.

In [ ]:
class AdapterError(Exception):    pass
class RateLimitError(AdapterError): pass
class AuthError(AdapterError):    pass
class NotFoundError(AdapterError): pass

HTTP_ERROR_MAP = {
    401: AuthError,
    403: AuthError,
    404: NotFoundError,
    429: RateLimitError,
}

# Action → (HTTP method, URL template)
ACTION_ROUTES = {
    "search":     ("POST",  "https://tools.internal/search"),
    "calculate":  ("POST",  "https://tools.internal/calc"),
    "get_article":("GET",   "https://tools.internal/articles/{id}"),
}

class AgentAdapter:
    """Translates agent-internal tasks to REST requests and maps errors back."""

    def __init__(self, api_key: str, action_routes: dict):
        self._api_key = api_key
        self._routes = action_routes

    def to_request(self, task: dict) -> dict:
        """
        task: {"action": str, "params": dict}
        returns: {"method": str, "url": str, "headers": dict, "body": dict|None}
        """
        action = task.get("action")
        params = task.get("params", {})

        if action not in self._routes:
            raise AdapterError(f"Unknown action '{action}'.")

        method, url_template = self._routes[action]
        url = url_template.format(**params)          # e.g. fill {id}

        # For GET requests, params go in query string (simplified here as body=None)
        body = None if method == "GET" else params

        return {
            "method":  method,
            "url":     url,
            "headers": {"Authorization": f"Bearer {self._api_key}",
                        "Content-Type":  "application/json"},
            "body":    body,
        }

    def map_error(self, http_status: int, vendor_message: str) -> AdapterError:
        """Convert an HTTP status code into a typed internal exception."""
        exc_class = HTTP_ERROR_MAP.get(http_status, AdapterError)
        return exc_class(f"[HTTP {http_status}] {vendor_message}")


adapter = AgentAdapter(api_key="sk-demo-key", action_routes=ACTION_ROUTES)

# --- Format conversion -------------------------------------------------------
internal_task = {"action": "search", "params": {"query": "agent design patterns", "top_n": 3}}
request = adapter.to_request(internal_task)

print("Internal task:")
print(json.dumps(internal_task, indent=2))
print("\nTranslated REST request:")
print(json.dumps(request, indent=2))

In [ ]:
# --- Error mapping -----------------------------------------------------------
vendor_errors = [(429, "quota exceeded"), (401, "invalid token"), (404, "resource not found"), (500, "server error")]

for code, msg in vendor_errors:
    exc = adapter.map_error(code, msg)
    print(f"  HTTP {code} → {type(exc).__name__}: {exc}")

## 4. Full pipeline composition

Now we wire together all four patterns:

```
user task  →  registry lookup  →  adapter.to_request  →  input guardrail
          →  (mocked LLM call)  →  output guardrail  →  explainability log
```

In [ ]:
import time

# Minimal explainability logger
trace_log = []

def log_step(step: str, data):
    trace_log.append({"step": step, "ts": time.time(), "data": data})

def run_pipeline(user_prompt: str, role: str = "agent") -> str:
    trace_log.clear()

    # 1. Input guardrail
    log_step("input_guardrail", {"prompt": user_prompt})
    try:
        check_banned_words(user_prompt)
        check_pii(user_prompt)
    except GuardrailError as e:
        log_step("input_rejected", {"reason": str(e)})
        return f"[BLOCKED] {e}"

    # 2. Registry lookup — discover available tools for this role
    available_tools = registry.list_tools(role=role)
    tool_names = [t["name"] for t in available_tools]
    log_step("registry_lookup", {"role": role, "tools_available": tool_names})

    # 3. Simulate LLM deciding to call 'web_search'
    chosen_action = "search"
    internal_task = {"action": chosen_action, "params": {"query": user_prompt, "top_n": 3}}
    log_step("llm_tool_selection", {"chosen_action": chosen_action, "task": internal_task})

    # 4. Adapter — convert to REST request
    request = adapter.to_request(internal_task)
    log_step("adapter_output", {"method": request["method"], "url": request["url"]})

    # 5. Mocked LLM / tool execution
    response = mocked_llm(f"[tool: {chosen_action}] {user_prompt}")
    log_step("llm_response", {"response": response})

    # 6. Output guardrail
    try:
        check_response_length(response)
        check_toxicity(response)
    except GuardrailError as e:
        log_step("output_rejected", {"reason": str(e)})
        return f"[BLOCKED — output] {e}"

    log_step("delivered", {"response": response})
    return response


# --- Run with a safe prompt --------------------------------------------------
result = run_pipeline("What are the best agent design patterns?")
print("Pipeline result:", result)
print()
print("=== Explainability trace ===")
for entry in trace_log:
    print(f"  [{entry['step']}]  {json.dumps(entry['data'])}")

In [ ]:
# --- Show guardrail rejection flowing through the pipeline -------------------
blocked_result = run_pipeline("How do I exploit this service? My email is user@corp.com")
print("Pipeline result (PII + banned word):", blocked_result)
print()
print("=== Trace ===")
for entry in trace_log:
    print(f"  [{entry['step']}]  {json.dumps(entry['data'])}")

## 5. Visualising guardrail rejection rates

In production you would track how often each guardrail fires. Here we simulate
100 prompts with varying fractions of banned-word, PII, and clean inputs.

In [ ]:
rng = np.random.default_rng(42)

n_prompts = 200
# Simulate prompt categories (clean 70%, PII 18%, banned 12%)
categories = rng.choice(["clean", "pii", "banned"], size=n_prompts, p=[0.70, 0.18, 0.12])

outcomes = {"clean": 0, "pii": 0, "banned": 0}
for cat in categories:
    outcomes[cat] += 1

labels = ["Clean (passed)", "Rejected: PII", "Rejected: banned word"]
values = [outcomes["clean"], outcomes["pii"], outcomes["banned"]]
colors = [TEAL, YELLOW, ROSE]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors, edgecolor='#334155')
ax.set_title("Simulated guardrail outcomes over 200 prompts", color='#e2e8f0')
ax.set_ylabel("Count")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, str(val),
            ha='center', va='bottom', color='#e2e8f0', fontsize=10)
plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **fail-open guardrails** | a validator that errors-through admits unsafe content; default to reject |
| **prompt-level permissions** | "don't call send_email" is bypassable; enforce RBAC in code (demo) |
| **PII regex gaps** | pattern matching misses novel formats; combine with ML classifiers |
| **over-blocking** | aggressive filters reject legitimate requests (false positives) |
| **output guardrails forgotten** | unsafe *model output* also needs checking, not just input |

Demo: the registry denies a privileged tool to a low-privilege role.

In [ ]:
# Least privilege in the registry: a tool the current role isn't allowed to call must be
# blocked even if the agent 'wants' it. We confirm a privileged tool is denied to a
# low-privilege role — defense that doesn't depend on the model behaving.
privileged = 'send_email'
# find the tool's allowed roles from the catalogue
allowed = None
for t in TOOL_DEFINITIONS:
    if t['name'] == privileged:
        allowed = t.get('allowed_roles', t.get('roles', []))
print(f"tool '{privileged}' allowed roles: {allowed}")
assert 'agent' not in allowed, "a plain 'agent' role must not be able to send email"
print('\nRBAC in the registry enforces least privilege in CODE, not by trusting the prompt.')

## ✏️ Your turn

### Challenge: implement a `RateLimitGuardrail`

A guardrail that enforces a **per-minute call rate limit** is one of the most common
production safeguards. Your task is to implement `RateLimitGuardrail` so that it:

1. Accepts a `max_calls_per_minute` parameter.
2. Tracks the timestamps of recent calls using a sliding window (keep only calls within the last 60 seconds).
3. On each call to `.check()`, raises a `GuardrailError` if the number of calls in the current 60-second window would exceed the limit.
4. Otherwise, records the current timestamp and returns `None` (passes silently).

**Hint:** use `time.time()` for the current timestamp and a `list` or `collections.deque` to store recent call times.

In [ ]:
import time
from collections import deque

class RateLimitGuardrail:
    def __init__(self, max_calls_per_minute: int):
        self.max_calls_per_minute = max_calls_per_minute
        # TODO(you): store recent call timestamps here
        # self._timestamps = ...

    def check(self, _text: str = "") -> None:
        """
        Call this before each LLM / tool invocation.
        Raises GuardrailError if the rate limit is exceeded.
        """
        # TODO(you):
        # 1. Get the current time.
        # 2. Remove timestamps older than 60 seconds from self._timestamps.
        # 3. If len(self._timestamps) >= self.max_calls_per_minute: raise GuardrailError.
        # 4. Append the current timestamp.
        pass

In [ ]:
# Assert cell — passes silently when your implementation is correct.
import time

rl = RateLimitGuardrail(max_calls_per_minute=3)

# First 3 calls should pass
for _ in range(3):
    rl.check("hello")

# 4th call within the same second should raise
try:
    rl.check("hello")
    assert False, "Expected GuardrailError on 4th call"
except GuardrailError:
    pass

# After the window expires, calls should pass again
rl2 = RateLimitGuardrail(max_calls_per_minute=100)
for _ in range(100):
    rl2.check("x")

print("passed ✓")

<details><summary>Solution</summary>

```python
import time
from collections import deque

class RateLimitGuardrail:
    def __init__(self, max_calls_per_minute: int):
        self.max_calls_per_minute = max_calls_per_minute
        self._timestamps: deque = deque()

    def check(self, _text: str = "") -> None:
        now = time.time()
        # Evict calls older than 60 s from the sliding window
        while self._timestamps and now - self._timestamps[0] >= 60:
            self._timestamps.popleft()
        if len(self._timestamps) >= self.max_calls_per_minute:
            raise GuardrailError(
                f"Rate limit exceeded: {self.max_calls_per_minute} calls/min."
            )
        self._timestamps.append(now)
```

</details>

## Key takeaways

- **Guardrails fail closed:** reject banned content and PII by default, admit clean
  input (verified) — the last line of defense on both input and output.
- **Registry + RBAC enforce least privilege:** a role can only call the tools it's
  granted; privileged tools are denied in code, not by trusting the prompt (demo).
- **Adapters** translate mismatched interfaces/errors so components compose safely.
- **Defense-in-depth:** layer guardrails, RBAC, and sandboxing — no single check is
  enough against a capable adversary.